**pre-requisites**

In [ ]:
#pip install numpy matplotlib


In [ ]:
import glob
import os
import time

file_list = glob.glob(os.path.join("instances", "*.tsp"))

results = {
    os.path.basename(instance): {
        "Ablation1": {"PathLength": None, "TimeElapsed": None},
        "Ablation2": {"PathLength": None, "TimeElapsed": None},
        "Ablation3": {"PathLength": None, "TimeElapsed": None},
        "Ablation4": {"PathLength": None, "TimeElapsed": None},
        "Ablation5": {"PathLength": None, "TimeElapsed": None},
        "Ablation6": {"PathLength": None, "TimeElapsed": None},
    }
    for instance in file_list
}

best_path_overall = None
best_path_length_overall = float('inf')

**2Opt + ACO**

In [ ]:
import numpy as np
import random
import glob
import os

def read(filename):
    infile = open(filename, 'r')

    Name = infile.readline().strip().split()[2]  # NAME
    FileType = infile.readline().strip().split()[2]  # TYPE
    Comment = infile.readline().strip().split()[2]  # COMMENT
    Dimension = infile.readline().strip().split()[2]  # DIMENSION
    EdgeWeightType = infile.readline().strip().split()[2]  # EDGE_WEIGHT_TYPE
    infile.readline()

    print("Solving:", Name)
    print("Dimension:", Dimension)

    nodelist = np.empty(shape=(int(Dimension), 2))
    N = int(Dimension)
    for i in range(0, int(Dimension)):
        x, y = infile.readline().strip().split()[1:]
        nodelist[i] = [float(x), float(y)]

    infile.close()
    return nodelist

def calculate_path_length(path):
    path_length = 0
    for i in range(len(path) - 1):
        current_city = path[i]
        next_city = path[i + 1]
        path_length += distances[current_city][next_city]
    path_length += distances[path[-1]][path[0]]
    return path_length

def local_search_2opt(path):
    best_path = path
    best_path_length = calculate_path_length(path)

    improved = True
    while improved:
        improved = False
        for i in range(1, num_cities - 2):
            for j in range(i + 1, num_cities):
                if j - i == 1:
                    continue
                new_path = path[:i] + path[i:j][::-1] + path[j:]
                new_path_length = calculate_path_length(new_path)
                if new_path_length < best_path_length:
                    best_path = new_path
                    best_path_length = new_path_length
                    improved = True
        path = best_path

    return best_path

def choose_next_city(current_city, allowed_cities):
    if random.random() < epsilon:
        return random.choice(allowed_cities)
    else:
        pheromone_levels = pheromones[current_city, allowed_cities]
        return allowed_cities[np.argmax(pheromone_levels)]

def update_pheromones(paths):
    global pheromones
    delta_pheromones = np.zeros((num_cities, num_cities))
    for path in paths:
        path_length = calculate_path_length(path)
        rewards = 1.0 / path_length
        for i in range(len(path) - 1):
            current_city = path[i]
            next_city = path[i + 1]
            delta_pheromones[current_city][next_city] += rewards

    pheromones = (1 - alpha) * pheromones + delta_pheromones

alpha = 0.3
epsilon = 0.1
num_iterations = 100
num_ants = 20
num_local_search_iterations = 20

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time()  
    cities = read(filename)
    num_cities = len(cities)

    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

    pheromones = np.ones((num_cities, num_cities))

    best_path_overall = None
    best_path_length_overall = float('inf')

    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = []

        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited_cities = set([current_city])
            path = [current_city]
            path_length = 0

            while len(visited_cities) < num_cities:
                allowed_cities = list(set(range(num_cities)) - visited_cities)
                next_city = choose_next_city(current_city, allowed_cities)
                visited_cities.add(next_city)
                path.append(next_city)
                path_length += distances[current_city][next_city]
                current_city = next_city

            path_length += distances[path[-1]][path[0]]
            if path_length < best_path_length:
                best_path_length = path_length
                best_path = path

        update_pheromones([best_path])

        best_path = local_search_2opt(best_path)
        best_path_length = calculate_path_length(best_path)

        for _ in range(num_local_search_iterations):
            new_path = local_search_2opt(best_path)
            new_path_length = calculate_path_length(new_path)
            if new_path_length < best_path_length:
                best_path = new_path
                best_path_length = new_path_length

        if best_path_length < best_path_length_overall:
            best_path_length_overall = best_path_length
            best_path_overall = best_path

    print("Optimal Path (Q-learning Ablated):", best_path_overall)
    print("Optimal Path Length (Q-learning Ablated):",best_path_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")
    print("------------------------------------------------------------")
    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{1}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }


**Q-learning + 2opt**

In [ ]:
import numpy as np
import random
import glob

def read(filename):
    Name = "Unknown"
    Dimension = 0
    with open(filename, 'r') as infile:
        for line in infile:
            line = line.strip()
            if line.startswith("NAME"):
                parts = line.split(":")
                if len(parts) > 1:
                    Name = parts[1].strip()
            elif line.startswith("DIMENSION"):
                parts = line.split(":")
                if len(parts) > 1:
                    Dimension = int(parts[1].strip())
            elif line.startswith("NODE_COORD_SECTION"):
                break
        nodelist = np.empty((Dimension, 2))
        for i in range(Dimension):
            coords_line = infile.readline().strip().split()
            nodelist[i] = [float(coords_line[1]), float(coords_line[2])]
    return Name, nodelist

def calculate_path_length(path, distances):
    length = 0
    for i in range(len(path)-1):
        length += distances[path[i]][path[i+1]]
    length += distances[path[-1]][path[0]]
    return length

def local_search_2opt(path, distances):
    best_path = path
    best_path_length = calculate_path_length(path, distances)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(path)-2):
            for j in range(i+1, len(path)):
                if j - i == 1:
                    continue
                new_path = path[:i] + path[i:j][::-1] + path[j:]
                new_length = calculate_path_length(new_path, distances)
                if new_length < best_path_length:
                    best_path = new_path
                    best_path_length = new_length
                    improved = True
        path = best_path
    return best_path

alpha = 0.3
epsilon = 0.1
num_iterations = 100
num_ants = 20
num_local_search_iterations = 20

def choose_next_city(current_city, allowed_cities, q_table):
    if random.random() < epsilon:
        return random.choice(allowed_cities)
    else:
        q_values = q_table[current_city, allowed_cities]
        return allowed_cities[np.argmax(q_values)]

def update_q_table(q_table, path, rewards):
    for i in range(len(path)-1):
        c = path[i]
        n = path[i+1]
        q_table[c,n] += alpha*(rewards - q_table[c,n])

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time()  

    Name, cities = read(filename)
    num_cities = len(cities)
    print(f"Solving instance: {Name} (File: {filename})")
    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    q_table = np.zeros((num_cities, num_cities))
    best_path_overall = None
    best_length_overall = float('inf')
    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = None
        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited = {current_city}
            path = [current_city]
            while len(visited) < num_cities:
                allowed = list(set(range(num_cities)) - visited)
                next_city = choose_next_city(current_city, allowed, q_table)
                visited.add(next_city)
                path.append(next_city)
                current_city = next_city
            length = calculate_path_length(path, distances)
            if length < best_path_length:
                best_path = path
                best_path_length = length
            update_q_table(q_table, path, 1.0/length)
        best_path = local_search_2opt(best_path, distances)
        best_path_length = calculate_path_length(best_path, distances)
        for _ in range(num_local_search_iterations):
            new_path = local_search_2opt(best_path, distances)
            new_length = calculate_path_length(new_path, distances)
            if new_length < best_path_length:
                best_path = new_path
                best_path_length = new_length
        if best_path_length < best_length_overall:
            best_length_overall = best_path_length
            best_path_overall = best_path
    print("Optimal Path (Q-learning+2-OPT):", best_path_overall)
    print("Optimal Path Length (Q-learning+2-OPT):", best_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")

    print("------------------------------------------------------------")
    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{2}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }


**ACO + Q-learning**

In [ ]:
import numpy as np
import random
import glob

def read(filename):
    Name = "Unknown"
    Dimension = 0
    with open(filename, 'r') as infile:
        for line in infile:
            line = line.strip()
            if line.startswith("NAME"):
                parts = line.split(":")
                if len(parts) > 1:
                    Name = parts[1].strip()
            elif line.startswith("DIMENSION"):
                parts = line.split(":")
                if len(parts) > 1:
                    Dimension = int(parts[1].strip())
            elif line.startswith("NODE_COORD_SECTION"):
                break
        nodelist = np.empty((Dimension, 2))
        for i in range(Dimension):
            coords_line = infile.readline().strip().split()
            nodelist[i] = [float(coords_line[1]), float(coords_line[2])]
    return Name, nodelist

def calculate_path_length(path, distances):
    length = 0
    for i in range(len(path)-1):
        length += distances[path[i]][path[i+1]]
    length += distances[path[-1]][path[0]]
    return length

alpha = 0.3
epsilon = 0.1
num_iterations = 100
num_ants = 20

def choose_next_city(current_city, allowed_cities, q_table):
    if random.random() < epsilon:
        return random.choice(allowed_cities)
    else:
        q_values = q_table[current_city, allowed_cities]
        return allowed_cities[np.argmax(q_values)]

def update_q_table(q_table, path, rewards):
    for i in range(len(path)-1):
        c = path[i]
        n = path[i+1]
        q_table[c,n] += alpha*(rewards - q_table[c,n])

def update_pheromones(paths, distances):
    global pheromones
    delta = np.zeros((num_cities, num_cities))
    for path in paths:
        length = calculate_path_length(path, distances)
        reward = 1.0/length
        for i in range(len(path)-1):
            delta[path[i]][path[i+1]] += reward
    pheromones = (1 - alpha)*pheromones + delta

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time()  
    Name, cities = read(filename)
    num_cities = len(cities)
    print(f"Solving instance: {Name} (File: {filename})")
    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    q_table = np.zeros((num_cities, num_cities))
    pheromones = np.ones((num_cities, num_cities))
    best_path_overall = None
    best_length_overall = float('inf')
    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = None
        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited = {current_city}
            path = [current_city]
            while len(visited) < num_cities:
                allowed = list(set(range(num_cities)) - visited)
                next_city = choose_next_city(current_city, allowed, q_table)
                visited.add(next_city)
                path.append(next_city)
                current_city = next_city
            length = calculate_path_length(path, distances)
            if length < best_path_length:
                best_path_length = length
                best_path = path
            update_q_table(q_table, path, 1.0/length)
        update_pheromones([best_path], distances)
        if best_path_length < best_length_overall:
            best_length_overall = best_path_length
            best_path_overall = best_path
    print("Optimal Path (ACO+Q-learning):", best_path_overall)
    print("Optimal Path Length (ACO+Q-learning):", best_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")

    print("------------------------------------------------------------")
    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{3}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }


**Q-learning only**

In [ ]:
import numpy as np
import random
import glob

def read(filename):
    Name = "Unknown"
    Dimension = 0
    with open(filename, 'r') as infile:
        for line in infile:
            line = line.strip()
            if line.startswith("NAME"):
                parts = line.split(":")
                if len(parts) > 1:
                    Name = parts[1].strip()
            elif line.startswith("DIMENSION"):
                parts = line.split(":")
                if len(parts) > 1:
                    Dimension = int(parts[1].strip())
            elif line.startswith("NODE_COORD_SECTION"):
                break
        nodelist = np.empty((Dimension, 2))
        for i in range(Dimension):
            coords_line = infile.readline().strip().split()
            nodelist[i] = [float(coords_line[1]), float(coords_line[2])]
    return Name, nodelist

def calculate_path_length(path, distances):
    length = 0
    for i in range(len(path)-1):
        length += distances[path[i]][path[i+1]]
    length += distances[path[-1]][path[0]]
    return length

alpha = 0.3
epsilon = 0.1
num_iterations = 100
num_ants = 20

def choose_next_city(current_city, allowed_cities, q_table):
    if random.random() < epsilon:
        return random.choice(allowed_cities)
    else:
        q_values = q_table[current_city, allowed_cities]
        return allowed_cities[np.argmax(q_values)]

def update_q_table(q_table, path, rewards):
    for i in range(len(path)-1):
        c = path[i]
        n = path[i+1]
        q_table[c,n] += alpha*(rewards - q_table[c,n])

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time() 
    Name, cities = read(filename)
    num_cities = len(cities)
    print(f"Solving instance: {Name} (File: {filename})")
    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    q_table = np.zeros((num_cities, num_cities))
    best_path_overall = None
    best_length_overall = float('inf')
    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = None
        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited = {current_city}
            path = [current_city]
            while len(visited) < num_cities:
                allowed = list(set(range(num_cities)) - visited)
                next_city = choose_next_city(current_city, allowed, q_table)
                visited.add(next_city)
                path.append(next_city)
                current_city = next_city
            length = calculate_path_length(path, distances)
            if length < best_path_length:
                best_path_length = length
                best_path = path
            update_q_table(q_table, path, 1.0 / length)
        if best_path_length < best_length_overall:
            best_length_overall = best_path_length
            best_path_overall = best_path
    print("Optimal Path (Q-learning Alone):", best_path_overall)
    print("Optimal Path Length (Q-learning Alone):", best_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")

    print("------------------------------------------------------------")
    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{4}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }


**2-opt only**

In [ ]:
import numpy as np
import random
import glob

def read(filename):
    Name = "Unknown"
    Dimension = 0
    with open(filename, 'r') as infile:
        for line in infile:
            line = line.strip()
            if line.startswith("NAME"):
                parts = line.split(":")
                if len(parts) > 1:
                    Name = parts[1].strip()
            elif line.startswith("DIMENSION"):
                parts = line.split(":")
                if len(parts) > 1:
                    Dimension = int(parts[1].strip())
            elif line.startswith("NODE_COORD_SECTION"):
                break
        nodelist = np.empty((Dimension, 2))
        for i in range(Dimension):
            coords_line = infile.readline().strip().split()
            nodelist[i] = [float(coords_line[1]), float(coords_line[2])]
    return Name, nodelist

def calculate_path_length(path, distances):
    length = 0
    for i in range(len(path)-1):
        length += distances[path[i]][path[i+1]]
    length += distances[path[-1]][path[0]]
    return length

def local_search_2opt(path, distances):
    best_path = path
    best_path_length = calculate_path_length(path, distances)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(path)-2):
            for j in range(i+1, len(path)):
                if j - i == 1:
                    continue
                new_path = path[:i] + path[i:j][::-1] + path[j:]
                new_length = calculate_path_length(new_path, distances)
                if new_length < best_path_length:
                    best_path = new_path
                    best_path_length = new_length
                    improved = True
        path = best_path
    return best_path

num_iterations = 100
num_ants = 20
num_local_search_iterations = 20

def choose_next_city(current_city, allowed_cities):
    return random.choice(allowed_cities)

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time()  

    Name, cities = read(filename)
    num_cities = len(cities)
    print(f"Solving instance: {Name} (File: {filename})")
    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    best_path_overall = None
    best_length_overall = float('inf')
    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = None
        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited = {current_city}
            path = [current_city]
            while len(visited) < num_cities:
                allowed = list(set(range(num_cities)) - visited)
                next_city = choose_next_city(current_city, allowed)
                visited.add(next_city)
                path.append(next_city)
                current_city = next_city
            path_length = calculate_path_length(path, distances)
            if path_length < best_path_length:
                best_path = path
                best_path_length = path_length
        best_path = local_search_2opt(best_path, distances)
        best_path_length = calculate_path_length(best_path, distances)
        for _ in range(num_local_search_iterations):
            new_path = local_search_2opt(best_path, distances)
            new_length = calculate_path_length(new_path, distances)
            if new_length < best_path_length:
                best_path = new_path
                best_path_length = new_length
        if best_path_length < best_length_overall:
            best_length_overall = best_path_length
            best_path_overall = best_path
    print("Optimal Path (2-OPT Alone):", best_path_overall)
    print("Optimal Path Length (2-OPT Alone):", best_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")

    print("------------------------------------------------------------")

    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{5}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }

**ACO only**

In [ ]:
import numpy as np
import random
import glob




def read(filename):
    Name = "Unknown"
    Dimension = 0
    with open(filename, 'r') as infile:
        for line in infile:
            line = line.strip()
            if line.startswith("NAME"):
                parts = line.split(":")
                if len(parts) > 1:
                    Name = parts[1].strip()
            elif line.startswith("DIMENSION"):
                parts = line.split(":")
                if len(parts) > 1:
                    Dimension = int(parts[1].strip())
            elif line.startswith("NODE_COORD_SECTION"):
                break
        nodelist = np.empty((Dimension, 2))
        for i in range(Dimension):
            coords_line = infile.readline().strip().split()
            nodelist[i] = [float(coords_line[1]), float(coords_line[2])]
    return Name, nodelist

def calculate_path_length(path, distances):
    length = 0
    for i in range(len(path)-1):
        length += distances[path[i]][path[i+1]]
    length += distances[path[-1]][path[0]]
    return length

alpha = 0.3
epsilon = 0.1
num_iterations = 100
num_ants = 20

def choose_next_city(current_city, allowed_cities):
    if random.random() < epsilon:
        return random.choice(allowed_cities)
    else:
        pheromone_levels = pheromones[current_city, allowed_cities]
        return allowed_cities[np.argmax(pheromone_levels)]

def update_pheromones(paths, distances):
    global pheromones
    delta = np.zeros((num_cities, num_cities))
    for path in paths:
        length = calculate_path_length(path, distances)
        reward = 1.0 / length
        for i in range(len(path)-1):
            delta[path[i]][path[i+1]] += reward
    pheromones = (1 - alpha) * pheromones + delta

for filename in glob.glob(os.path.join("instances", "*.tsp")):
    start_time = time.time()  

    Name, cities = read(filename)
    num_cities = len(cities)
    print(f"Solving instance: {Name} (File: {filename})")
    distances = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            x1, y1 = cities[i]
            x2, y2 = cities[j]
            distances[i][j] = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    pheromones = np.ones((num_cities, num_cities))
    best_path_overall = None
    best_length_overall = float('inf')
    for iteration in range(num_iterations):
        best_path_length = float('inf')
        best_path = None
        for ant in range(num_ants):
            current_city = random.randint(0, num_cities - 1)
            visited = {current_city}
            path = [current_city]
            while len(visited) < num_cities:
                allowed = list(set(range(num_cities)) - visited)
                next_city = choose_next_city(current_city, allowed)
                visited.add(next_city)
                path.append(next_city)
                current_city = next_city
            length = calculate_path_length(path, distances)
            if length < best_path_length:
                best_path_length = length
                best_path = path
        update_pheromones([best_path], distances)
        if best_path_length < best_length_overall:
            best_length_overall = best_path_length
            best_path_overall = best_path
    print("Optimal Path (ACO Alone):", best_path_overall)
    print("Optimal Path Length (ACO Alone):", best_length_overall)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    print(f"Time Elapsed for {instance_name}: {elapsed_time:.2f} seconds")
    print("------------------------------------------------------------")
    instance_name = os.path.basename(filename)  # Extract file name
    results[instance_name][f"Ablation{6}"] = {
    "PathLength": best_length_overall,
    "TimeElapsed": elapsed_time
    }


**Summary**

In [ ]:
import pandas as pd

# Extract Path Lengths
path_length_data = {
    instance: {ablation: results[instance][ablation]["PathLength"] for ablation in results[instance]}
    for instance in results
}

# Extract Time Elapsed
time_elapsed_data = {
    instance: {ablation: results[instance][ablation]["TimeElapsed"] for ablation in results[instance]}
    for instance in results
}

# Convert to DataFrames
path_length_df = pd.DataFrame(path_length_data).T  # Transpose for readability
time_elapsed_df = pd.DataFrame(time_elapsed_data).T  # Transpose for readability

# Rename Columns for Clarity
path_length_df.columns = ["Ablation1", "Ablation2", "Ablation3", "Ablation4", "Ablation5", "Ablation6"]
time_elapsed_df.columns = ["Ablation1", "Ablation2", "Ablation3", "Ablation4", "Ablation5", "Ablation6"]

# Set Index Name
path_length_df.index.name = "Instance Name"
time_elapsed_df.index.name = "Instance Name"

# Display Path Length Table
print("\nPath Length Table:")
print(path_length_df)

# Display Time Elapsed Table
print("\nTime Elapsed Table:")
print(time_elapsed_df)

# Save both tables to CSV files
path_length_df.to_csv("tsp_path_lengths.csv")
time_elapsed_df.to_csv("tsp_time_elapsed.csv")

print("\nPath lengths saved to 'tsp_path_lengths.csv'.")
print("Time elapsed saved to 'tsp_time_elapsed.csv'.")
